# Provider verification

Discovers every model each configured provider serves, sends one tiny chat to each, and shows what works. The same logic backs `tsp-ai verify`. Keys come from `.env` (`<PLATFORM>_KEY`). **Never print keys.**

In [ ]:
from pathlib import Path

from tsp.ai import LLMClient
from tsp.ai.discovery import verify

env = Path('.env') if Path('.env').exists() else Path('../../../.env')
client = LLMClient.from_env(env)
print('Registered providers:', sorted(client._router._providers))

## Probe

`free_only=True` skips models not known to be free, so no billing surprises. Set it to `False` to probe everything.

In [ ]:
catalog = await verify(client, free_only=True, concurrency=2, on_provider=lambda r: print(r.platform, r.key_status.value, f'{len(r.ok_models)}/{len(r.models)} ok'))

## Results

In [ ]:
rows = [
    (r.platform, m.id, m.status.value, m.latency_ms, m.error)
    for r in catalog.providers.values()
    for m in r.models
]
try:
    import pandas as pd

    display(pd.DataFrame(rows, columns=['provider', 'model', 'status', 'latency_ms', 'error']))
except ImportError:
    for row in rows:
        print(*row)

## Save and use

The catalog is what `LLMClient(catalog=...)` and `tsp-ai serve` route over.

In [ ]:
path = catalog.save()
print('Saved', path)
await client.close()